# M3C-002B-CAL-001 开发校准训练
T4、Internet、已有 GITHUB_TOKEN + KAGGLE_API_TOKEN。请 Save Version → Save & Run All，不运行未保存的交互草稿。
三臂各32768个查询Targets，Seed27101，FP32；正式训练/测试/PPL关闭。真实运行ID缺失时首更新前停止并发布诊断。


In [ ]:
from pathlib import Path
import os, signal, subprocess, sys, time

CALIBRATION_TAG = 'M3C-002B-CAL-001:c362efd864e893a9d06df296f06d0f01c27c8735c610bfed3dfeb5a36ca93211'
BRANCH = 'research/m3c-002b-cal-001'
REPOSITORY = 'https://github.com/ArcheLabs/mini-cells.git'
WORKSPACE = Path('/kaggle/working')
REPO = WORKSPACE / 'minicells-m3c-002b-cal-001'
OUTPUT = WORKSPACE / 'm3c-002b-cal-001-bootstrap'
HARD_LIMIT_SECONDS = 7200
started = time.monotonic()
PLATFORM_RESERVE_SECONDS = 60
deadline = started + HARD_LIMIT_SECONDS - PLATFORM_RESERVE_SECONDS

def run_bounded(command, cwd=WORKSPACE, timeout_seconds=None):
    remaining = deadline - time.monotonic()
    if timeout_seconds is not None:
        remaining = min(remaining, timeout_seconds)
    if remaining <= 1:
        raise TimeoutError('The two-hour notebook budget is exhausted.')
    process = subprocess.Popen(command, cwd=cwd, text=True, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, start_new_session=(os.name == 'posix'))
    try:
        output, _ = process.communicate(timeout=remaining)
    except subprocess.TimeoutExpired:
        if os.name == 'posix': os.killpg(process.pid, signal.SIGTERM)
        else: process.terminate()
        try: output, _ = process.communicate(timeout=5)
        except subprocess.TimeoutExpired:
            if os.name == 'posix': os.killpg(process.pid, signal.SIGKILL)
            else: process.kill()
            output, _ = process.communicate()
        print(output or '')
        raise TimeoutError('Notebook time limit reached; partial reports and final models, if any, remains under ' + str(OUTPUT))
    print(output or '')
    if process.returncode != 0:
        raise RuntimeError('Command failed (' + str(process.returncode) + '). Output is under ' + str(OUTPUT))

if REPO.exists():
    if not (REPO / '.git').exists():
        import shutil
        shutil.rmtree(REPO)
if REPO.exists():
    origin = subprocess.run(['git', '-C', str(REPO), 'remote', 'get-url', 'origin'], capture_output=True, text=True)
    if origin.returncode != 0 or 'ArcheLabs/mini-cells' not in origin.stdout:
        raise RuntimeError(f'Refusing to use an unrelated directory: {REPO}')
    run_bounded(['git', '-C', str(REPO), 'fetch', '--depth=1', 'origin', BRANCH])
    run_bounded(['git', '-C', str(REPO), 'switch', '-C', BRANCH, 'FETCH_HEAD'])
    run_bounded(['git', '-C', str(REPO), 'clean', '-fd'])
else:
    run_bounded(['git', 'clone', '--depth=1', '--single-branch', '--branch', BRANCH,
        REPOSITORY, str(REPO)])

SOURCE_COMMIT = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
OUTPUT = WORKSPACE / ('m3c-002b-cal-001-' + SOURCE_COMMIT[:12])

CANDIDATE = WORKSPACE / 'mamba3-minimal-reference'
PIN = '15ba9983123703d9f4f7d6f8709152d0a07edbd9'
if not CANDIDATE.exists():
    run_bounded(['git','clone','--depth=1','--no-checkout','https://github.com/VikramLex/mamba3-minimal.git',str(CANDIDATE)],timeout_seconds=180)
run_bounded(['git','-C',str(CANDIDATE),'fetch','--depth=1','origin',PIN],timeout_seconds=120)
run_bounded(['git','-C',str(CANDIDATE),'checkout','--detach',PIN])
import importlib.util
if importlib.util.find_spec('einops') is None:
    run_bounded([sys.executable,'-m','pip','install','einops'],timeout_seconds=120)
run_bounded([sys.executable,str(REPO / 'research/projects/cellular-language-models/experiments/mamba3-cellular-series/M3C-002B-CAL-001/run.py'),'--output',str(OUTPUT),'--candidate',str(CANDIDATE),'--started',str(started)],cwd=REPO,timeout_seconds=6600)
